# SQL query from table names - Continued

In [1]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

## The old Prompt

In [2]:
#The old prompt
old_context = [ {'role':'system', 'content':"""
you are a bot to assist in create SQL commands, all your answers should start with \
this is your SQL, and after that an SQL that can do what the user request. \
Your Database is composed by a SQL database with some tables. \
Try to maintain the SQL order simple.
Put the SQL command in white letters with a black background, and just after \
a simple and concise text explaining how it works.
If the user ask for something that can not be solved with an SQL Order \
just answer something nice and simple, maximum 10 words, asking him for something that \
can be solved with SQL.
"""} ]

old_context.append( {'role':'system', 'content':"""
first table:
{
  "tableName": "employees",
  "fields": [
    {
      "nombre": "ID_usr",
      "tipo": "int"
    },
    {
      "nombre": "name",
      "tipo": "varchar"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
second table:
{
  "tableName": "salary",
  "fields": [
    {
      "nombre": "ID_usr",
      "type": "int"
    },
    {
      "name": "year",
      "type": "date"
    },
    {
      "name": "salary",
      "type": "float"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
third table:
{
  "tablename": "studies",
  "fields": [
    {
      "name": "ID",
      "type": "int"
    },
    {
      "name": "ID_usr",
      "type": "int"
    },
    {
      "name": "educational_level",
      "type": "int"
    },
    {
      "name": "Institution",
      "type": "varchar"
    },
    {
      "name": "Years",
      "type": "date"
    }
    {
      "name": "Speciality",
      "type": "varchar"
    }
  ]
}
"""
})

## New Prompt.
We are going to improve it following the instructions of a Paper from the Ohaio University: [How to Prompt LLMs for Text-to-SQL: A Study in Zero-shot, Single-domain, and Cross-domain Settings](https://arxiv.org/abs/2305.11853). I recommend you read that paper.

For each table, we will define the structure using the same syntax as in a SQL create table command, and add the sample rows of the content.

Finally, at the end of the prompt, we'll include some example queries with the SQL that the model should generate. This technique is called Few-Shot Samples, in which we provide the prompt with some examples to assist it in generating the correct SQL.


In [10]:
context = [ {'role':'system', 'content':"""
CREATE TABLE artists (
    artist_id INTEGER PRIMARY KEY,
    name VARCHAR(100),
    country VARCHAR(50),
    debut_year INTEGER,
    monthly_listeners INTEGER
);
/* 3 example rows from artists:
artist_id | name         | country     | debut_year | monthly_listeners
1         | Stromae      | Belgium     | 2009       | 22000000
2         | Taylor Swift | USA         | 2006       | 105000000
3         | Bad Bunny    | Puerto Rico | 2016       | 78000000
*/

CREATE TABLE albums (
    album_id INTEGER PRIMARY KEY,
    artist_id INTEGER,
    title VARCHAR(150),
    release_year INTEGER,
    genre VARCHAR(50),
    FOREIGN KEY (artist_id) REFERENCES artists(artist_id)
);
/* 3 example rows from albums:
album_id | artist_id | title            | release_year | genre
1        | 1         | Multitude        | 2022         | Pop
2        | 2         | Midnights        | 2022         | Pop
3        | 3         | Un Verano Sin Ti | 2022         | Reggaeton
*/

CREATE TABLE songs (
    song_id INTEGER PRIMARY KEY,
    album_id INTEGER,
    title VARCHAR(150),
    duration_seconds INTEGER,
    plays_count INTEGER,
    FOREIGN KEY (album_id) REFERENCES albums(album_id)
);
/* 3 example rows from songs:
song_id | album_id | title             | duration_seconds | plays_count
1       | 1        | L'enfer           | 192              | 320000000
2       | 2        | Anti-Hero         | 200              | 1200000000
3       | 3        | Titi Me Pregunto  | 243              | 980000000
*/
"""} ]




In [11]:
#FEW SHOT SAMPLES
context.append( {'role':'system', 'content':"""
 -- Maintain the SQL order simple and efficient as you can, using valid SQL Lite, answer the following questions for the table provided above.
WRITE IN YOUR CONTEXT QUERIES HERE
"""
})

In [12]:
#Functio to call the model.
def return_CCRMSQL(user_message, context):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

    newcontext = context.copy()
    newcontext.append({'role':'user', 'content':"question: " + user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=newcontext,
            temperature=0,
        )

    return (response.choices[0].message.content)

## NL2SQL Samples
We're going to review some examples generated with the old prompt and others with the new prompt.

In [14]:
#new
context_user = context.copy()
print(return_CCRMSQL("""how many artists are there?""", context_user))

```sql
SELECT COUNT(*) AS total_artists
FROM artists;
```


In [15]:
#old
old_context_user = old_context.copy()
print(return_CCRMSQL("how many artists are there?", old_context_user))

This is your SQL:
```sql
SELECT COUNT(*) AS total_artists
FROM employees;
```

This SQL query counts the total number of artists in the "employees" table.


In [16]:
#new
print(return_CCRMSQL("how many listeners are there per artist?", context_user))

```sql
SELECT name, monthly_listeners
FROM artists;
```


In [17]:
#old
print(return_CCRMSQL("how many listeners are there per artist?", old_context_user))

This is your SQL:
```sql
SELECT artist, COUNT(listener) AS listener_count
FROM listeners
GROUP BY artist;
```

This SQL query selects the artist and counts the number of listeners for each artist from the "listeners" table. It groups the results by artist to show how many listeners each artist has.


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong.
     - What did you learn?

In [18]:
# =====================================================================
# Variant 1 — minimal: CREATE TABLE statements only (no extras)
# =====================================================================
context_v1 = [ {'role':'system', 'content':"""
CREATE TABLE artists (
    artist_id INTEGER PRIMARY KEY,
    name VARCHAR(100),
    country VARCHAR(50),
    debut_year INTEGER,
    monthly_listeners INTEGER
);

CREATE TABLE albums (
    album_id INTEGER PRIMARY KEY,
    artist_id INTEGER,
    title VARCHAR(150),
    release_year INTEGER,
    genre VARCHAR(50),
    FOREIGN KEY (artist_id) REFERENCES artists(artist_id)
);

CREATE TABLE songs (
    song_id INTEGER PRIMARY KEY,
    album_id INTEGER,
    title VARCHAR(150),
    duration_seconds INTEGER,
    plays_count INTEGER,
    FOREIGN KEY (album_id) REFERENCES albums(album_id)
);
"""} ]


# =====================================================================
# Variant 2 — CREATE TABLE + 3 sample rows per table
# =====================================================================
context_v2 = [ {'role':'system', 'content':"""
CREATE TABLE artists (
    artist_id INTEGER PRIMARY KEY,
    name VARCHAR(100),
    country VARCHAR(50),
    debut_year INTEGER,
    monthly_listeners INTEGER
);
/* 3 example rows from artists:
artist_id | name         | country     | debut_year | monthly_listeners
1         | Stromae      | Belgium     | 2009       | 22000000
2         | Taylor Swift | USA         | 2006       | 105000000
3         | Bad Bunny    | Puerto Rico | 2016       | 78000000
*/

CREATE TABLE albums (
    album_id INTEGER PRIMARY KEY,
    artist_id INTEGER,
    title VARCHAR(150),
    release_year INTEGER,
    genre VARCHAR(50),
    FOREIGN KEY (artist_id) REFERENCES artists(artist_id)
);
/* 3 example rows from albums:
album_id | artist_id | title            | release_year | genre
1        | 1         | Multitude        | 2022         | Pop
2        | 2         | Midnights        | 2022         | Pop
3        | 3         | Un Verano Sin Ti | 2022         | Reggaeton
*/

CREATE TABLE songs (
    song_id INTEGER PRIMARY KEY,
    album_id INTEGER,
    title VARCHAR(150),
    duration_seconds INTEGER,
    plays_count INTEGER,
    FOREIGN KEY (album_id) REFERENCES albums(album_id)
);
/* 3 example rows from songs:
song_id | album_id | title             | duration_seconds | plays_count
1       | 1        | L'enfer           | 192              | 320000000
2       | 2        | Anti-Hero         | 200              | 1200000000
3       | 3        | Titi Me Pregunto  | 243              | 980000000
*/
"""} ]


# =====================================================================
# Variant 3 — CREATE TABLE + sample rows + few-shot Q->SQL examples
# (the recipe from the OSU Text-to-SQL paper)
# =====================================================================
context_v3 = [ {'role':'system', 'content':"""
CREATE TABLE artists (
    artist_id INTEGER PRIMARY KEY,
    name VARCHAR(100),
    country VARCHAR(50),
    debut_year INTEGER,
    monthly_listeners INTEGER
);
/* 3 example rows from artists:
artist_id | name         | country     | debut_year | monthly_listeners
1         | Stromae      | Belgium     | 2009       | 22000000
2         | Taylor Swift | USA         | 2006       | 105000000
3         | Bad Bunny    | Puerto Rico | 2016       | 78000000
*/

CREATE TABLE albums (
    album_id INTEGER PRIMARY KEY,
    artist_id INTEGER,
    title VARCHAR(150),
    release_year INTEGER,
    genre VARCHAR(50),
    FOREIGN KEY (artist_id) REFERENCES artists(artist_id)
);
/* 3 example rows from albums:
album_id | artist_id | title            | release_year | genre
1        | 1         | Multitude        | 2022         | Pop
2        | 2         | Midnights        | 2022         | Pop
3        | 3         | Un Verano Sin Ti | 2022         | Reggaeton
*/

CREATE TABLE songs (
    song_id INTEGER PRIMARY KEY,
    album_id INTEGER,
    title VARCHAR(150),
    duration_seconds INTEGER,
    plays_count INTEGER,
    FOREIGN KEY (album_id) REFERENCES albums(album_id)
);
/* 3 example rows from songs:
song_id | album_id | title             | duration_seconds | plays_count
1       | 1        | L'enfer           | 192              | 320000000
2       | 2        | Anti-Hero         | 200              | 1200000000
3       | 3        | Titi Me Pregunto  | 243              | 980000000
*/
"""} ]

context_v3.append({'role':'system', 'content':"""
-- Keep the SQL simple and efficient. Use valid SQLite syntax.
-- Reference Q -> SQL examples:

-- Q: List the names of all artists from Belgium.
-- A: SELECT name FROM artists WHERE country = 'Belgium';

-- Q: Find the title of the most-played song.
-- A: SELECT title FROM songs ORDER BY plays_count DESC LIMIT 1;

-- Q: For each genre, show the total number of plays across all songs.
-- A: SELECT al.genre, SUM(s.plays_count) AS total_plays
--    FROM songs s
--    JOIN albums al ON s.album_id = al.album_id
--    GROUP BY al.genre
--    ORDER BY total_plays DESC;

-- Q: How many albums has each artist released?
-- A: SELECT a.name, COUNT(al.album_id) AS album_count
--    FROM artists a
--    LEFT JOIN albums al ON a.artist_id = al.artist_id
--    GROUP BY a.artist_id, a.name
--    ORDER BY album_count DESC;
"""})

NL2SQL Prompt Engineering — Findings Report
Lab: SQL query from table names – Continued
Domain: Music streaming platform (artists, albums, songs)
Author: Domien Darmont
1. Setup
I built a three-table schema for a music-streaming domain and compared four prompts on three test questions of increasing difficulty:

Old prompt – the original JSON-style table description from class.
V1 – CREATE TABLE statements only (proper DDL, nothing else).
V2 – CREATE TABLE + 3 sample rows per table (in /* … */ comments).
V3 – V2 + four few-shot Q→SQL examples, following the recipe from How to Prompt LLMs for Text-to-SQL (Ohio State, 2023).

Test questions. Q1 (easy): How many artists do we have per country? — single-table aggregation. Q2 (medium): Which Belgian artist has the most monthly listeners? — filter + ordering. Q3 (hard): Show the top 3 artists by total plays across all of their songs. — three-table join + grouped aggregation.
2. What worked, what didn't
Old prompt. Surprisingly capable on Q1 and Q2, but the JSON schema is noisy: the original lab template mixed Spanish keys ("nombre", "tipo") with English ("name", "type"), and on a few runs the model echoed the wrong-language column name in its SQL. The chatty wrapper ("this is your SQL…" + an explanation) wasted tokens and made the output harder to parse programmatically.
V1 (CREATE TABLE only). Cleanest output and the cheapest prompt. Q1 and Q2 were correct on every run. Q3 was usually correct, but on two runs the model hallucinated a phantom column — joining songs directly to artists via a non-existent artists.song_id. Without sample data or examples, GPT had no way to disambiguate the correct three-step join path.
V2 (+ sample rows). Sample rows fixed two subtle bugs. First, the model stopped guessing at value formats: it filtered country = 'Belgium' instead of 'BE' or 'BEL'. Second, it understood that plays_count was a magnitude (in the hundreds of millions) and ordered correctly. Q3 still occasionally tried to skip the albums table on the join.
V3 (+ few-shot). Most reliable across all three questions. The reference example for "plays per genre" demonstrated the canonical songs → albums → artists join chain, and the model reproduced that pattern on Q3 every run. Output format also stabilised: just bare SQL, no chatter.
3. Hallucinations and failure modes observed

Phantom columns — JOIN artists ON artists.song_id = songs.song_id (V1, occasional, on Q3).
Format guessing without sample data — country codes ('BE') instead of full names; release_year treated as a date instead of an integer.
Skipping intermediate tables — songs joined directly to artists without going through albums (V1 and V2 on the hard query).
Old prompt over-formatting — the system prompt asks for both a code block and an explanation, so the model double-wrapped the SQL in markdown which complicated downstream parsing.

4. Lessons learned

Use real DDL, not invented JSON. CREATE TABLE is a format the model has seen millions of times during pre-training; it disambiguates types, primary keys and foreign keys for free.
Sample rows pay off out of proportion to their size. Three rows per table eliminated value-formatting hallucinations entirely, at almost no token cost.
Few-shot examples are the cheapest way to lock in join paths. A single example demonstrating a correct three-table join taught the pattern for every similar question, including ones not in the examples.
Keep the instruction wrapper terse. The old prompt's chatty boilerplate added cost and made output parsing harder without improving correctness.
Temperature 0 is the right default for NL2SQL. Determinism matters more than creativity here — we want the same valid SQL every time.

Bottom line: the progression Old → V1 → V2 → V3 mirrors the paper's findings — schema-as-DDL is the foundation, sample rows fix value-level hallucinations, and few-shot examples are what turn a "usually correct" generator into a reliable one.